# ATLAS IMAGE AI (Colab)
1. **Ambiente de execução → Alterar o tipo de ambiente → T4 GPU** → Salvar.
2. **Ambiente de execução → Executar tudo**. Na primeira vez baixa cerca de 7 GB.
3. No fim da última célula aparece um link `https://....gradio.live`. Abra no celular e deixe a célula rodando.

Prompts e imagens passam pelos servidores do Google e pelo relé do Gradio. Confira a licença de cada modelo no cartão dele; as restrições de uso continuam valendo. Não use "Salvar cópia no GitHub" com a saída da execução, porque ela contém o link público.

In [ ]:
!pip -q install diffusers transformers accelerate safetensors gradio sentencepiece sacremoses bitsandbytes
import torch
assert torch.cuda.is_available(), "Sem GPU. Ative em Ambiente de execução → Alterar o tipo de ambiente → T4 GPU e execute de novo."
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import gc, os, random, time, inspect, warnings
warnings.filterwarnings("ignore")
import torch, gradio as gr
from PIL.PngImagePlugin import PngInfo
from diffusers import AutoPipelineForText2Image
from diffusers.utils import logging as dlog
dlog.set_verbosity_error()

MODELS = {
    "SDXL 1.0 base · equilibrado": {"repo": "stabilityai/stable-diffusion-xl-base-1.0", "kind": "sdxl"},
    "FLUX.1 schnell · melhor fidelidade ao prompt (experimental, download grande)": {"repo": "black-forest-labs/FLUX.1-schnell", "kind": "flux"},
    "SDXL-Turbo · rápido, 512 px": {"repo": "stabilityai/sdxl-turbo", "kind": "turbo"},
}
SIZES = {
    "1:1 · 1024×1024": (1024, 1024), "3:4 · 896×1152": (896, 1152), "4:3 · 1152×896": (1152, 896),
    "9:16 · 768×1344": (768, 1344), "16:9 · 1344×768": (1344, 768),
}
PRESETS = {
    "Fotografia": "photo, 35mm, natural light, sharp focus, high detail",
    "Retrato": "portrait, soft studio light, shallow depth of field, detailed face",
    "Paisagem": "wide landscape, golden hour, dramatic sky, high detail",
    "Anime": "anime style, clean line art, vibrant colors",
    "Ilustração": "digital illustration, bold shapes, painterly, detailed",
    "Fantasia": "fantasy art, epic, magical atmosphere, intricate details",
    "Arte conceitual": "concept art, cinematic composition, matte painting",
    "Render 3D": "3d render, global illumination, ultra detailed",
}
LANGS = ["Português (traduzir para inglês)", "Inglês (enviar como está)"]
EXAMPLES = [
    ["um gato laranja dormindo em uma poltrona de veludo verde, luz suave da janela"],
    ["foto profissional de uma xícara de café sobre uma mesa de mármore, iluminação de estúdio"],
    ["montanhas com neve ao nascer do sol, lago com reflexo, cinematográfico"],
    ["retrato de uma cientista sorrindo em um laboratório moderno"],
]
OUT = "/content/atlas_outputs"
os.makedirs(OUT, exist_ok=True)
cur = {"name": None, "pipe": None}
stop = {"flag": False}
tr = {"tok": None, "mdl": None}

def to_en(text):
    if not text.strip():
        return text
    if tr["mdl"] is None:
        from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
        n = "Helsinki-NLP/opus-mt-ROMANCE-en"
        tr["tok"], tr["mdl"] = AutoTokenizer.from_pretrained(n), AutoModelForSeq2SeqLM.from_pretrained(n)
    b = tr["tok"]([text], return_tensors="pt", truncation=True, max_length=256)
    out = tr["mdl"].generate(**b, max_new_tokens=256, num_beams=4)
    return tr["tok"].decode(out[0], skip_special_tokens=True)

def load(name):
    m = MODELS[name]
    if m["kind"] == "flux":
        from diffusers import FluxPipeline, FluxTransformer2DModel, BitsAndBytesConfig as DQ
        from transformers import T5EncoderModel, BitsAndBytesConfig as TQ
        t = FluxTransformer2DModel.from_pretrained(m["repo"], subfolder="transformer", torch_dtype=torch.float16,
            quantization_config=DQ(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16))
        t5 = T5EncoderModel.from_pretrained(m["repo"], subfolder="text_encoder_2", torch_dtype=torch.float16,
            quantization_config=TQ(load_in_8bit=True))
        p = FluxPipeline.from_pretrained(m["repo"], transformer=t, text_encoder_2=t5, torch_dtype=torch.float16)
        p.enable_model_cpu_offload()
        return p
    return AutoPipelineForText2Image.from_pretrained(m["repo"], torch_dtype=torch.float16, variant="fp16", use_safetensors=True).to("cuda")

def get_pipe(name):
    if cur["name"] == name:
        return cur["pipe"]
    cur["pipe"], cur["name"] = None, None
    gc.collect(); torch.cuda.empty_cache()
    cur["pipe"], cur["name"] = load(name), name
    return cur["pipe"]

def generate(model, prompt, lang, negative, size, steps, cfg, seed, count, hist, progress=gr.Progress()):
    prompt, negative = (prompt or "").strip(), (negative or "").strip()
    if not prompt:
        raise gr.Error("Escreva um prompt.")
    stop["flag"] = False
    notes, kind = [], MODELS[model]["kind"]
    sent, sent_neg = prompt, negative
    if lang == LANGS[0]:
        progress(0, desc="Traduzindo para inglês…")
        try:
            sent, sent_neg = to_en(prompt), (to_en(negative) if negative else "")
        except Exception as e:
            notes.append(f"Tradução indisponível ({e}). Enviei o texto original.")
    progress(0, desc="Carregando o modelo…")
    try:
        pipe = get_pipe(model)
    except Exception as e:
        raise gr.Error(f"Não foi possível carregar este modelo: {e}")
    w, h = SIZES[size]
    steps, cfg, count = int(steps), float(cfg), int(count)
    extra = {}
    if kind == "turbo":
        w, h = w // 2, h // 2
        if cfg != 0 or sent_neg or steps > 4:
            notes.append("SDXL-Turbo: sem orientação, sem prompt negativo e no máximo 4 etapas (valores ajustados).")
        cfg, sent_neg, steps = 0.0, "", min(steps, 4)
    elif kind == "flux":
        if cfg != 0 or sent_neg or steps > 8:
            notes.append("FLUX schnell: sem orientação, sem prompt negativo e 4 etapas (valores ajustados).")
        cfg, sent_neg, steps = 0.0, "", (steps if steps <= 8 else 4)
        extra = {"max_sequence_length": 256}
    else:
        n = len(pipe.tokenizer(sent).input_ids)
        if n > 77:
            notes.append(f"Atenção: o prompt tem {n} tokens e este modelo lê só 77. O final foi cortado.")
        extra = {"negative_prompt": sent_neg or None}
    base = random.randint(0, 2**32 - 1) if seed is None or int(seed) < 0 else int(seed)
    shown, lines, t0 = [], [], time.time()
    try:
        for i in range(count):
            s = (base + i) % 2**32
            def tick(p, step, t, kw, i=i):
                progress((step + 1, steps), desc=f"Imagem {i + 1}/{count}")
                return kw
            gen = torch.Generator("cpu" if kind == "flux" else "cuda").manual_seed(s)
            img = pipe(prompt=sent, width=w, height=h, num_inference_steps=steps, guidance_scale=cfg,
                       generator=gen, callback_on_step_end=tick, **extra).images[0]
            if stop["flag"]:
                notes.append("Interrompido pelo usuário.")
                break
            d = (f"prompt original: {prompt}\nprompt enviado ao modelo: {sent}\nnegativo enviado: {sent_neg}\n"
                 f"modelo: {model} | {w}x{h} | etapas {steps} | CFG {cfg} | seed {s}")
            meta = PngInfo(); meta.add_text("parameters", d)
            img.save(f"{OUT}/atlas_{int(time.time())}_{s}.png", pnginfo=meta)
            shown.append((img, f"seed {s}")); lines.append(d)
    except torch.cuda.OutOfMemoryError:
        gc.collect(); torch.cuda.empty_cache()
        raise gr.Error("Memória da GPU insuficiente. Use uma resolução menor ou o SDXL-Turbo.")
    new_hist = (shown + (hist or []))[:40]
    return shown, "\n\n".join(lines + notes) + f"\n\nTempo: {time.time() - t0:.1f} s", new_hist, new_hist

def do_stop():
    stop["flag"] = True
    if cur["pipe"] is not None:
        cur["pipe"]._interrupt = True

def add_preset(p, k):
    p = (p or "").strip()
    return p if not k else (f"{p}, {PRESETS[k]}" if p else PRESETS[k])

print("Carregando o modelo padrão (pode demorar alguns minutos)…")
get_pipe(list(MODELS)[0])

CSS = """
.gradio-container{max-width:1100px!important;margin:auto}
#hero{padding:22px 4px 8px}
#hero h1{margin:0;font-size:2rem;letter-spacing:.12em;font-weight:800;background:linear-gradient(90deg,#8b5cf6,#3b82f6);-webkit-background-clip:text;background-clip:text;color:transparent}
#hero p{margin:.3rem 0 0;opacity:.75}
#go{min-height:52px;font-size:1.05rem;font-weight:700}
footer{display:none!important}
"""
try:
    THEME = gr.themes.Base(primary_hue="violet", secondary_hue="blue", neutral_hue="slate", font=["system-ui", "sans-serif"]).set(
        body_background_fill="#0f1016", body_background_fill_dark="#0f1016",
        block_background_fill="#171822", block_background_fill_dark="#171822",
        block_border_color="#2a2b3d", block_border_color_dark="#2a2b3d",
        body_text_color="#ececf4", body_text_color_dark="#ececf4",
        input_background_fill="#1e1f2e", input_background_fill_dark="#1e1f2e",
        button_primary_background_fill="linear-gradient(90deg,#8b5cf6,#3b82f6)",
        button_primary_background_fill_dark="linear-gradient(90deg,#8b5cf6,#3b82f6)",
        button_primary_text_color="white", button_primary_text_color_dark="white")
except Exception:
    THEME = gr.themes.Soft(primary_hue="violet", secondary_hue="blue")
NEW = "theme" in inspect.signature(gr.Blocks.launch).parameters
bk = {} if NEW else {"theme": THEME, "css": CSS}
lk = {"theme": THEME, "css": CSS} if NEW else {}

ABOUT = """### Dicas para imagens fiéis ao prompt
- Descreva **um assunto principal** com objeto, cenário, luz e estilo. Evite pedir muitos elementos de uma vez.
- Modelos de imagem erram contagens, textos escritos e posições exatas. O FLUX schnell erra menos nisso, mas é mais lento.
- Com "Português", o app traduz e mostra o texto enviado em "O que foi processado". Se quiser controle total, escreva em inglês e escolha "Inglês".
- Use a mesma seed para repetir uma imagem e mude só uma palavra por vez para ajustar.

### Privacidade e licenças
Prompts e imagens passam pelos servidores do Google (Colab) e pelo relé do Gradio. Este app não adiciona filtros de palavras nem altera o prompt além da tradução escolhida por você. As licenças e as restrições de uso de cada modelo continuam valendo; confira no cartão de cada um."""

with gr.Blocks(title="ATLAS IMAGE AI", **bk) as demo:
    gr.HTML('<div id="hero"><h1>ATLAS IMAGE AI</h1><p>Estúdio de criação de imagens por IA</p></div>')
    hist = gr.State([])
    with gr.Tabs():
        with gr.Tab("Criar"):
            with gr.Row():
                with gr.Column(scale=5):
                    prompt = gr.Textbox(label="Prompt", lines=4, placeholder="Descreva o assunto, o cenário, a luz e o estilo")
                    lang = gr.Radio(LANGS, value=LANGS[0], label="Idioma do prompt")
                    with gr.Row():
                        preset = gr.Dropdown(list(PRESETS), value=None, label="Preset de estilo", scale=3)
                        add = gr.Button("Aplicar", scale=1)
                    negative = gr.Textbox(label="Prompt negativo (só SDXL base)", lines=2)
                    with gr.Accordion("Ajustes avançados", open=False):
                        model = gr.Dropdown(list(MODELS), value=list(MODELS)[0], label="Modelo")
                        size = gr.Dropdown(list(SIZES), value=list(SIZES)[0], label="Proporção e resolução")
                        with gr.Row():
                            steps = gr.Slider(1, 50, value=30, step=1, label="Etapas")
                            cfg = gr.Slider(0, 15, value=7, step=0.5, label="Orientação (CFG)")
                        with gr.Row():
                            seed = gr.Number(value=-1, precision=0, label="Seed (-1 = aleatória)")
                            count = gr.Slider(1, 4, value=1, step=1, label="Quantidade")
                    go = gr.Button("Gerar imagem", variant="primary", elem_id="go")
                    halt = gr.Button("Interromper")
                with gr.Column(scale=6):
                    gallery = gr.Gallery(label="Resultado", columns=2, height=520)
                    info = gr.Textbox(label="O que foi processado", lines=7)
            gr.Examples(EXAMPLES, inputs=[prompt], label="Exemplos")
        with gr.Tab("Histórico"):
            hgal = gr.Gallery(label="Imagens desta sessão", columns=3, height=600)
        with gr.Tab("Sobre"):
            gr.Markdown(ABOUT)
    add.click(add_preset, [prompt, preset], prompt)
    go.click(generate, [model, prompt, lang, negative, size, steps, cfg, seed, count, hist], [gallery, info, hist, hgal])
    halt.click(do_stop, None, None)

demo.queue().launch(share=True, debug=True, **lk)